In [16]:
# === Baseline "aktuelles Regelwerk" (geloggter PSP), Reihenfolge nach attempt_number ===
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# ---------------- Config ----------------
DF_PATH   = Path("df.pkl")   # dein Datensatz
TEST_SIZE = 0.2
RNG       = 42
ALPHA     = 0.05             # Business: Erfolg - α * Kosten
SMOOTH_M  = 20               # Laplace-Glättung für PSP-Raten (Estimated)

# ---------------- Load & prep ----------------
df = pd.read_pickle(DF_PATH).replace([np.inf, -np.inf], np.nan).copy()

required = {"transaction_id", "PSP", "success", "fee_successful", "fee_not_successful", "attempt_number"}
missing = required - set(df.columns)
assert not missing, f"Fehlende Spalten: {missing}"

# Datentypen / Sauberkeit
df["PSP"] = df["PSP"].astype("string").str.strip().fillna("missing")
df["success"] = pd.to_numeric(df["success"], errors="coerce").fillna(0).astype(int)
df["fee_successful"] = pd.to_numeric(df["fee_successful"], errors="coerce").fillna(0.0)
df["fee_not_successful"] = pd.to_numeric(df["fee_not_successful"], errors="coerce").fillna(0.0)
df["attempt_number"] = pd.to_numeric(df["attempt_number"], errors="coerce").fillna(1).astype(int)

# Reihenfolge innerhalb der Session NUR nach attempt_number
df = df.sort_values(["transaction_id", "attempt_number"]).reset_index(drop=True)

# Realisierte Gebühr pro Versuch (für Observed)
df["fee_realized"] = df["success"]*df["fee_successful"] + (1 - df["success"])*df["fee_not_successful"]

# ---------------- Leakage-sicherer Split (nach transaction_id) ----------------
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RNG)
idx_tr, idx_te = next(gss.split(df, groups=df["transaction_id"]))
train_df = df.iloc[idx_tr].copy()
test_df  = df.iloc[idx_te].copy()

# ---------------- A) OBSERVED: Status quo mit echten Outcomes ----------------
obs = test_df.groupby("transaction_id").agg(
    success_final=("success", "max"),
    total_cost=("fee_realized", "sum"),
    n_attempts=("attempt_number","max"),
)
obs["business"] = obs["success_final"] - ALPHA * obs["total_cost"]

print("=== Baseline: aktuelles Regelwerk — OBSERVED (Session-Level) ===")
print(f"Success rate (observed)              : {obs['success_final'].mean():.6f}")
print(f"Average cost (observed)              : {obs['total_cost'].mean():.6f}")
print(f"Business score (α={ALPHA:.3f}, obs.) : {obs['business'].mean():.6f}")

# (Optional) Reporting: Ø-Gebühr getrennt nach Erfolg/Misserfolg
avg_fee_by_success = test_df.groupby("success")["fee_realized"].mean().rename({0:"not successful",1:"successful"})
print("Avg fee by success:")
for k, v in avg_fee_by_success.items():
    print(f"  - {k:16s}: {v:.6f}")

# ---------------- B) ESTIMATED: Nur mit Train-PSP-Raten (Laplace) ----------------
# p̂ je PSP aus TRAIN (keine Test-Labels verwenden)
prior = float(train_df["success"].mean())
psp_grp = train_df.groupby("PSP")["success"].agg(["sum", "count"])
psp_rate = (psp_grp["sum"] + SMOOTH_M * prior) / (psp_grp["count"] + SMOOTH_M)  # Laplace
rate_map = psp_rate.to_dict()

# p̂ & Fees je Versuch im TEST
test_df = test_df.sort_values(["transaction_id","attempt_number"]).copy()
test_df["p_hat"] = test_df["PSP"].map(rate_map).fillna(prior).astype(float)
test_df["fee_s"] = test_df["fee_successful"].astype(float)
test_df["fee_f"] = test_df["fee_not_successful"].astype(float)

def expected_session_kpi(session_rows: pd.DataFrame) -> pd.Series:
    """Erwartete Session-KPI unter geloggter Reihenfolge (attempt_number) & p̂ je Versuch."""
    s  = session_rows  # bereits sortiert
    p  = s["p_hat"].to_numpy(dtype=float)
    fs = s["fee_s"].to_numpy(dtype=float)
    ff = s["fee_f"].to_numpy(dtype=float)

    reach = 1.0
    exp_cost = 0.0
    p_final = 0.0
    for pi, fsi, ffi in zip(p, fs, ff):
        exp_cost += reach * (pi * fsi + (1 - pi) * ffi)
        p_final  += reach * pi
        reach    *= (1 - pi)
    business = p_final - ALPHA * exp_cost
    return pd.Series({"p_final": p_final, "exp_cost": exp_cost, "business": business})

est = test_df.groupby("transaction_id", sort=False).apply(expected_session_kpi)

print("\n=== Baseline: aktuelles Regelwerk — ESTIMATED (Session-Level, PSP-rate) ===")
print(f"Expected success (estimated)         : {est['p_final'].mean():.6f}")
print(f"Expected cost (estimated)            : {est['exp_cost'].mean():.6f}")
print(f"Business score (α={ALPHA:.3f}, est.): {est['business'].mean():.6f}")


=== Baseline: aktuelles Regelwerk — OBSERVED (Session-Level) ===
Success rate (observed)              : 0.267944
Average cost (observed)              : 2.325777
Business score (α=0.050, obs.) : 0.151656
Avg fee by success:
  - not successful  : 1.219553
  - successful      : 3.856509

=== Baseline: aktuelles Regelwerk — ESTIMATED (Session-Level, PSP-rate) ===
Expected success (estimated)         : 0.254651
Expected cost (estimated)            : 2.202128
Business score (α=0.050, est.): 0.144544
